# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints during backend development.

Default base URL is `http://localhost:3000`.

In [ ]:
import json
from typing import Any

import requests

In [ ]:
API_BASE = "http://localhost:3000"
TIMEOUT_SECONDS = 15

def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))

## 1) Health Check

In [ ]:
health = call_api("/health")
preview(health)

## 2) Tiles Endpoint
Adjust the viewport/filter params as needed.

In [ ]:
tiles_params = {
    "sw_lat": 51.48,
    "sw_lng": -0.22,
    "ne_lat": 51.54,
    "ne_lng": -0.06,
    "zoom": 13,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "confidence": 1,
    "score_tier": 0,
}

tiles = call_api("/api/tiles", params=tiles_params)
preview(tiles)

## 3) Nearby Endpoint

In [ ]:
nearby_params = {
    "lat": 51.5074,
    "lng": -0.1278,
    "radius_m": 1000,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "confidence": 1,
    "rank_threshold": 0,
    "page": 1,
}

nearby = call_api("/api/nearby", params=nearby_params)
preview(nearby)

## 4) Place Detail Endpoint
Run the next cell after running nearby/tiles so you can pick a real place id.

In [ ]:
sample_place_id = nearby.get("data", [{}])[0].get("id") if isinstance(nearby, dict) else None
sample_place_id

In [ ]:
if not sample_place_id:
    raise ValueError("No place id available. Set sample_place_id manually and retry.")

place = call_api(f"/api/place/{sample_place_id}")
preview(place)